In [1]:
import os, torch
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig, set_seed
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset
from preprocessing import build_sequences, split, SERVICES

c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
set_seed(42)

# ── Config — tweak these per experiment ───────────────────────────────────────
MODEL_ID      = "google/gemma-2-2b"
CKPT_BASE     = Path("./checkpoints")
MAX_LEN       = 512    # tokens; reduce to 512 or 256 if you hit OOM
LORA_RANK     = 16     # ablate: 4, 8, 16, 32
LORA_ALPHA    = 32     # rule of thumb: 2 × rank
EPOCHS        = 3
BATCH_SIZE    = 1
GRAD_ACCUM    = 16     # effective batch = 16

In [5]:
# ── 4-bit quantization config (QLoRA) ─────────────────────────────────────────
def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,   # saves ~0.4 bits per param extra
    )

# ── Model + tokenizer loader ──────────────────────────────────────────────────
def load_base_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"    # required for causal LM training

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=get_bnb_config(),
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
    )
    # Required step before attaching LoRA to a quantized model
    model = prepare_model_for_kbit_training(model)
    return model, tokenizer

# ── LoRA adapter ──────────────────────────────────────────────────────────────
def attach_lora(model, rank: int = LORA_RANK):
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=rank * 2,
        # Target all attention projections for best coverage
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

# ── Dataset preparation ───────────────────────────────────────────────────────
def make_hf_dataset(train_seqs: list, tokenizer, max_len: int) -> Dataset:
    """
    Converts normal-only sequences into a HuggingFace Dataset.
    Labels = input_ids (causal LM: predict next token at every position).
    """
    def tokenize(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_len,
            padding="max_length",
        )
        enc["labels"] = enc["input_ids"].copy()
        return enc

    ds = Dataset.from_list([{"text": s["text"]} for s in train_seqs])
    ds = ds.map(tokenize, batched=True, remove_columns=["text"])
    ds.set_format("torch")
    return ds

# ── Training ──────────────────────────────────────────────────────────────────
def fine_tune(service: str, train_seqs: list, rank: int = LORA_RANK):
    """
    Fine-tunes LLM+LoRA on normal-only log sequences for a given service.
    Saves the LoRA adapter (not the full model) to checkpoints/<service>/r<rank>/
    """
    ckpt_dir = CKPT_BASE / service / f"r{rank}"
    if ckpt_dir.exists():
        print(f"  Checkpoint exists at {ckpt_dir} — skipping training.")
        return ckpt_dir

    print(f"\n  Loading base model...")
    model, tokenizer = load_base_model(MODEL_ID)
    model = attach_lora(model, rank)

    print(f"  Preparing dataset ({len(train_seqs)} sequences)...")
    dataset = make_hf_dataset(train_seqs, tokenizer, MAX_LEN)

    args = TrainingArguments(
        output_dir=str(ckpt_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        bf16=True,
        logging_steps=5,
        save_strategy="epoch",
        save_total_limit=1,         # keep only the best checkpoint
        load_best_model_at_end=False,
        report_to="none",           # swap to "wandb" for loss curve tracking
        run_name=f"{service}-r{rank}",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    print(f"  Training {service} (rank={rank}, epochs={EPOCHS})...")
    trainer.train()

    # Save only the lightweight LoRA adapter weights (~10-50 MB vs 16 GB)
    model.save_pretrained(str(ckpt_dir))
    tokenizer.save_pretrained(str(ckpt_dir))
    print(f"  Saved adapter to {ckpt_dir}")

    # Free VRAM before next service
    del model, trainer
    torch.cuda.empty_cache()

    return ckpt_dir

In [ ]:
for service in SERVICES:
        print(f"\n{'='*55}")
        print(f"  Service: {service}")
        print(f"{'='*55}")

        seqs = build_sequences(service)
        if not seqs:
            print(f"  No sequences found — skipping.")
            continue

        train_seqs, _ = split(seqs)

        if len(train_seqs) < 5:
            print(f"  Too few training sequences ({len(train_seqs)}) — skipping.")
            continue

        fine_tune(service, train_seqs, rank=LORA_RANK)


  Service: client
[client] 6154 sequences (114 normal, 6040 anomalous)
  Train (normal only): 57
  Test  (mixed):       3077  (57 normal, 3020 anomalous)

  Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:09<00:00, 30.45it/s]


trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438
  Preparing dataset (57 sequences)...


Map: 100%|██████████| 57/57 [00:00<00:00, 74.86 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training client (rank=16, epochs=3)...


...

Loading weights: 100%|██████████| 288/288 [00:10<00:00, 28.75it/s]


trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438
  Preparing dataset (57 sequences)...


Parameter 'function'=<function make_hf_dataset.<locals>.tokenize at 0x000001B1B377CF60> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Map: 100%|██████████| 57/57 [00:00<00:00, 118.43 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training code (rank=16, epochs=3)...


c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [7]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

GPU Available: True
GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Total VRAM (GB): 6.441992192
